## Prédiction de la gravité des accidents corporels

Par Adam Bernabeu et Lucy Neveux, groupe 3.

## Introduction

La sécurité routière demeure un enjeu majeur de santé publique. Malgré l'évolution constante des normes de sécurité automobile et des infrastructures, la compréhension des déterminants de la gravité d'un accident reste complexe.

L'objectif de ce projet est de développer un modèle de prédiction capable de prédire la gravité des blessures des usagers impliqués dans des accidents corporels en France, essentiellement en France métropolitaine. Au-delà de la simple prédiction, ce travail vise à identifier les variables critiques qui font basculer un accident vers une issue fatale. Pour cela, nous prenons en compte les différences entre les bases de données à travers les années, et exploitons ces différences pour obtenir une meilleure interprétation pratique des résultats.

Cette étude s'appuie sur les bases de données annuelles du [Bulletin d'Analyse des Accidents Corporels (BAAC)](https://www.data.gouv.fr/datasets/bases-de-donnees-annuelles-des-accidents-corporels-de-la-circulation-routiere-annees-de-2005-a-2024) pour les années 2005 à 2024. Nous avons fusionné quatre tables fondamentales :
- **Usagers** : Profil (âge, sexe, place dans le véhicule) et équipements de sécurité.

- **Véhicules** : Type de motorisation, catégorie et point de choc initial.

- **Lieux** : Type de route, tracé, état de la surface et infrastructure.

- **Caractéristiques** : Conditions atmosphériques, luminosité et localisation géographique.

Les données de 2005-2018 et de 2019-2024 présentent plusieurs différences fondamentales. En particulier, la donnée de vitesse maximale autorisée sur la route où l'accident a eu lieu est indisponible avant 2019. Nous allons étudier l'influence de ces différences sur les prédictions de notre modèle.

*N.B. : Le notebook a été conçu en utilisant Python 3.12.*

# Sommaire
* [I. Introduction](#introduction)
* [II. Analyse des données de 2019-2024](#données-2019-2024)
    * [Nettoyage des données](#nettoyage-des-données)
        * [Caractéristiques des accidents](#caractéristiques)
        * [Usagers](#usagers)
        * [Lieu de l'accident](#lieux)
        * [Véhicules impliqués](#véhicules)
        * [Fusion des jeux de données](#fusion-des-dataframes-usagers-caractéristiques-et-lieux)
    * [Cartographie et analyse géographique](#cartographie-des-accidents-et-analyse-géographique)
    * [Feature engineering](#feature-engineering-2019-2024)
* [III. Analyse des données de 2005-2021](#données-2005-2021)
    * [Nettoyage des données](#nettoyage-des-données)
    * [Feature engineering](#feature-engineering-2005-2018)
* [IV. Prédictions](#prédictions)
    * [Prédiction 2019-2024](#prédiction-2019-2024)
        * [Sans véhicules](#analyse-sans-véhicules)
        * [Avec véhicules (2024 exclu)](#analyse-avec-véhicules)
    * [Prédiction 2005-2018](#prédiction-2005-2018)
* [V. Conclusion générale](#conclusion-générale)

## Installation

In [ ]:
# !pip install -r requirements.txt

import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from modules import dataviz, engineer_features, process_vma, prediction, process
import statsmodels

# # Fix certificates errors on data load:
import certifi
import os
os.environ["SSL_CERT_FILE"] = certifi.where()

In [ ]:
# Téléchargement des dataframes depuis la base de données des BAAC.
urls_y = {
    2019 : {
        "carac" : "https://www.data.gouv.fr/api/1/datasets/r/e22ba475-45a3-46ac-a0f7-9ca9ed1e283a",
        "lieux" : "https://www.data.gouv.fr/api/1/datasets/r/2ad65965-36a1-4452-9c08-61a6c874e3e6",
        "usagers" : "https://www.data.gouv.fr/api/1/datasets/r/36b1b7b3-84b4-4901-9163-59ae8a9e3028",
        "vehic" : "https://www.data.gouv.fr/api/1/datasets/r/780cd335-5048-4bd6-a841-105b44eb2667"
    },
    2020 : {
        "carac" : "https://www.data.gouv.fr/api/1/datasets/r/07a88205-83c1-4123-a993-cba5331e8ae0",
        "lieux" : "https://www.data.gouv.fr/api/1/datasets/r/e85c41f7-d4ea-4faf-877f-ab69a620ce21",
        "usagers" : "https://www.data.gouv.fr/api/1/datasets/r/78c45763-d170-4d51-a881-e3147802d7ee",
        "vehic" : "https://www.data.gouv.fr/api/1/datasets/r/a66be22f-c346-49af-b196-71df24702250"
    },
    2021 : {
        "carac" : "https://www.data.gouv.fr/api/1/datasets/r/85cfdc0c-23e4-4674-9bcd-79a970d7269b",
        "lieux" : "https://www.data.gouv.fr/api/1/datasets/r/8a4935aa-38cd-43af-bf10-0209d6d17434",
        "usagers" : "https://www.data.gouv.fr/api/1/datasets/r/ba5a1956-7e82-41b7-a602-89d7dd484d7a",
        "vehic" : "https://www.data.gouv.fr/api/1/datasets/r/0bb5953a-25d8-46f8-8c25-b5c2f5ba905e"
    },
    2022: {
        "carac" : "https://www.data.gouv.fr/api/1/datasets/r/5fc299c0-4598-4c29-b74c-6a67b0cc27e7",
        "lieux" : "https://www.data.gouv.fr/api/1/datasets/r/a6ef711a-1f03-44cb-921a-0ce8ec975995",
        "usagers" : "https://www.data.gouv.fr/api/1/datasets/r/62c20524-d442-46f5-bfd8-982c59763ec8",
        "vehic" : "https://www.data.gouv.fr/api/1/datasets/r/c9742921-4427-41e5-81bc-f13af8bc31a0"
    },
    2023 : {
        "carac" : "https://www.data.gouv.fr/api/1/datasets/r/104dbb32-704f-4e99-a71e-43563cb604f2",
        "lieux" : "https://www.data.gouv.fr/api/1/datasets/r/8bef19bf-a5e4-46b3-b5f9-a145da4686bc",
        "usagers" : "https://www.data.gouv.fr/api/1/datasets/r/68848e2a-28dd-4efc-9d5f-d512f7dbe66f",
        "vehic" : "https://www.data.gouv.fr/api/1/datasets/r/146a42f5-19f0-4b3e-a887-5cd8fbef057b"
    },
    2024 : {
        "carac" : "https://www.data.gouv.fr/api/1/datasets/r/83f0fb0e-e0ef-47fe-93dd-9aaee851674a",
        "lieux" : "https://www.data.gouv.fr/api/1/datasets/r/228b3cda-fdfb-4677-bd54-ab2107028d2d",
        "usagers" : "https://www.data.gouv.fr/api/1/datasets/r/f57b1f58-386d-4048-8f78-2ebe435df868",
        # Jeu de données véhicules indisponible pour l'année 2024
    }
}

# Dictionnaire qui contiendra les dataframes pour chaque année en vue de la concaténation

dfs = {}

yrl = [2019,2020,2021,2022,2023,2024]
names = ["carac","lieux","usagers","vehic"]

for name in names:
    list_df = []
    for yr in yrl:
        if yr==2024 and name=="vehic":
            continue
        url = urls_y[yr][name]
        print(f'{name} - {yr}')
        df_yr = pd.read_csv(url,sep=";", storage_options={"verify": certifi.where()}) # Le séparateur utilisé est ";"
        df_yr.insert(1,"year",yr) # Colonne année pour distinguer les accidents entre années en vue des opérations de fusion de dataframes
        list_df.append(df_yr)

    concatenated_df_type = pd.concat(list_df,ignore_index=True) # On fait fi de l'index
    dfs[name] = concatenated_df_type # On associe le dataframe des trois années au type de dataframe

# Prend environ 40 secondes à tourner

# Données 2019-2024 <a name="données-2019-2024"></a>

## Préparation des données

Observons la longueur de chacun des dataframes :

In [ ]:
for n,content in dfs.items(): 
    print(n,len(content))

In [ ]:
# for name in names:
#     dfs[name].to_csv(f"data_interm/new/{name}_new.csv",index=False)

lieux_new = dfs["lieux"]
usagers_new = dfs["usagers"]
carac_new = dfs["carac"]
vehic_new = dfs["vehic"]

### Chargement des données en .csv

In [ ]:
names = ["carac","lieux","usagers","vehic"]

# lieux_new = pd.read_csv("data_interm/new/lieux_new.csv")
# usagers_new = pd.read_csv("data_interm/new/usagers_new.csv")
# carac_new = pd.read_csv("data_interm/new/carac_new.csv")
# vehic_new = pd.read_csv("data_interm/new/vehic_new.csv")

Comme le jeu de données véhicules n'est pas disponible pour l'année 2024, nous nous en occuperons en dernier.

Il est normal que le dataframe "usagers" soit le plus long : en effet, pour chaque accident corporel, on peut compter plusieurs victimes. Il est plus surprenant que "lieux" soit plus long que "carac". 
Observons les doublons :

In [ ]:
lieux_new["Num_Acc"].value_counts().head(20)

In [ ]:
lieux_new[lieux_new["Num_Acc"]==202300035508]

In [ ]:
sum(carac_new["Num_Acc"].value_counts()>1)

In [ ]:
if "an" in carac_new.columns:
    carac_new.drop(columns=["an"],inplace=True) # Colonne an redondante avec colonne year

"Lieux" indique les différentes voies liées à l'accident lorsqu'il a eu lieu dans une intersection complexe. Il contient par ailleurs différentes informations sur le lieu de l'accident (type de route **catr**, l'état de la surface **surf**, la vitesse maximale autorisée **vma**).

Les données les plus générales sont les "caractéristiques de l'accident", uniques pour chaque accident. Dans chaque accident, on a des données sur le lieu de l'accident et l'avant-accident qui est différent selon les acteurs impliqués. Enfin, la partie "usagers" comporte des informations sur chacun des usagers impliqués dans l'accident ; c'est le dataset le plus large.

## Nettoyage des données

### Caractéristiques



Observons le dataset "carac".

In [ ]:
carac_new.head(20)

#### Valeurs manquantes
Concentrons nous ici sur les valeurs manquantes.
On a une importante granularité pour le moment précis de l'accident. Observons s'il y'a des données manquantes.

In [ ]:
carac_new=carac_new.replace([-1,"-1"],np.nan)

In [ ]:
missing_counts=carac_new.isna().sum()
missing_counts[missing_counts>0]

Le volume de données manquantes pour lum, int, atm et col est négligeable. On peut donc imputer par le mode pour ne pas biaiser le modèle.

Nous avons deux colonnes "Num_Acc" et "Accident_Id" qui représentent la même chose ; la colonne a changé de nom avec les années. Vérifions qu'il n'y a pas de contradiction entre les deux colonnes, et qu'elles ne sont jamais toutes deux vides.

In [ ]:
both_empty = carac_new[carac_new['Num_Acc'].isna() & carac_new['Accident_Id'].isna()]
both_present = carac_new[carac_new['Num_Acc'].notna() & carac_new['Accident_Id'].notna()]
print(len(both_empty))
print(len(both_present))

On voit que Accident_Id et Num_Acc se complètent parfaitement. On peut donc fusionner Accident_Id et Num_Acc.

Comme on dispose déjà d'informations sur la configuration de la route et qu'on dispose de toutes les longitudes et latitudes, on peut supprimer la colonne "adresse", qui n'apporte rien de plus.


In [ ]:
carac_new = process.clean_carac(carac_new)

In [ ]:
missing_counts=carac_new.isna().sum()
missing_counts[missing_counts>0]

### Usagers

Le jeu de données usagers est primordial, car il contient notre variable d'intérêt.

>Pour rappel, **grav** peut prendre quatre valeurs : 
>* -1 = NA
>* 1 = Indemne
>* 2 = Tué
>* 3 = Blessé hospitalisé
>* 4 = Blessé léger

In [ ]:
sns.histplot(usagers_new["grav"])

In [ ]:
print(round((usagers_new["grav"]==-1).mean(),4)*100)

# 0.06% des données sont -1 (pas de renseignement) pour la gravité

Seules 0.095% des enregistrements ont des données manquantes sur la target variable "grav". Elles sont extrêmement minoritaires, nous pouvons donc les supprimer.

In [ ]:
usagers_new = usagers_new[usagers_new["grav"]!=-1]

On choisit un mapping ordinal plus cohérent de la gravité de l'accident corporel :

In [ ]:
mapping = {1:0, # Indemne
           2:3, # Tué
           3:2, # Blessé hospitalisé
           4:1, # Blessé léger
            }
usagers_new["grav_ord"] = usagers_new["grav"].map(mapping)
usagers_new = usagers_new.drop(columns=["grav"],errors='ignore')
sns.histplot(usagers_new["grav_ord"],discrete=True)

#### Valeurs manquantes

On nettoie les valeurs manquantes de usagers_new à partir du fichier source process.

In [ ]:
df_obj = usagers_new.select_dtypes(['object'])
usagers_new[df_obj.columns] = df_obj.apply(lambda x: x.str.strip())
usagers_new.replace([-1,"-1"], np.nan, inplace=True)

In [ ]:
missing_counts=usagers_new.isna().sum()
missing_counts[missing_counts>0]

On remarque que ACTP, qui correspond à l'action du piéton selon la typologie suivante : 
>Se déplaçant 
>* 0 - non renseigné ou sans objet 
>* 1 - Sens véhicule heurtant 
>* 2 - Sens inverse du véhicule 

>Divers  
>* 3 - Traversant 
>* 4 - Masqué 
>* 5 - Jouant – courant 
>* 6 - Avec animal 
>* 9 - Autre

contient des valeurs qui ne sont PAS numériques.

En effet, à partir de 2019, on a également -1 : non renseigné, A : Monte/Descend du véhicule, et B : inconnue.

En transforme simplement A en 7 et B en 8.

Pour "place", nous prenons en compte la catégorie d'usager "catu" ; s'il s'agit d'un conducteur, il est à la place 1. Si c'est un passager, on applique le mode (valeur la plus fréquente). Si c'est un piéton, il est à la place 10.

On nettoie les autres valeurs manquantes et on crée une nouvelle variable âge, à partir de l'année de naissance et de l'année de l'accident.

In [ ]:
usagers_new=process.clean_usagers(usagers_new)

### Lieux
Observons le dataset "lieux".

In [ ]:
lieux_new[lieux_new["Num_Acc"].duplicated(keep=False)].sort_values(by="Num_Acc").head(20)
#Keep=False conserve les duplicates, contrairement au défaut keep=First qui ne montre que la première ligne.

Nous supprimons la colonne voie, qui est bruitée, et peu informative. On supprime donc également v1 et v2, qui donnent l'adresse. On supprime de même les colonnes concernant les points de repère (bornes kilométriques : pr1 et pr2). En effet, nous disposons déjà des longitudes et des latitudes pour chaque enregistrement (cf. caractéristiques plus haut).

A contrario, la VMA (vitesse maximale autorisée), la catégorie de route (catr - autoroute, route urbaine...), l'état de la surface (surf - conditions de la route : verglas, pluie, neige), le régime de circulation (circ - bidirectionnel, unidirectionnel,...), le type d'infrastructure (infra - ponts, tunnels ou carrefours), situation de l'accident (situ - où a précisément eu lieu l'accident : sur la chaussée, sur la bande d'arrêt d'urgence,...), sont particulièrement importantes pour notre prédiction.

Les colonnes plan, prof, nbv, , larrout, vosp, donnent des informations précises sur la configuration des lieux de l'accident. En particulier, prof (topographie), plan (courbure de la route), larrout (largeur de la route) sont très intéressantes.

In [ ]:
lieux_new=lieux_new.drop(columns=["voie","v1","v2","pr","pr1"],errors="ignore")

In [ ]:
lieux_new[lieux_new["Num_Acc"]==202300000001]

#### Valeurs manquantes

D'après la description des données, toutes les valeurs -1 correspondent à des valeurs manquantes.

In [ ]:
df_obj = lieux_new.select_dtypes(['object'])
lieux_new[df_obj.columns] = df_obj.apply(lambda x: x.str.strip())
lieux_new.replace([-1,"-1"], np.nan, inplace=True)

Observons quelles variables sont concernées. Nous traiterons la VMA plus bas.

In [ ]:
missing_counts = lieux_new.isna().sum()
print(missing_counts[missing_counts>0],len(lieux_new))

Dans  le cas du jeu de données lieux, l'opération est plus délicate, car de nombreuses valeurs sont interdépendantes. Nous utiliserons la catégorie de route, jamais manquante, pour remplacer les valeurs manquantes de nbv, puis de VMA plus bas.
Pour rappel : 

Catégorie de route :
* 1 – Autoroute
* 2 – Route nationale
* 3 – Route Départementale
* 4 – Voie Communales
* 5 – Hors réseau public
* 6 – Parc de stationnement ouvert à la circulation publique
* 7 – Routes de métropole urbaine
* 9 – autre 

Nous supprimons la colonne lartpc, qui n'est présente que pour 92 enregistrements sur 196410.

On remplace les valeurs manquantes de nbv (nombre de voies), au nombre de 233 sur plus de 196000, par la médiane de la catégorie de route concernée.

On impute par le mode (valeur majoritaire) pour 'circ' (régime de circulation), 'vosp' (voie réservée), 'prof' (profil), 'plan' (courbure), 'infra' (infrastructure) et 'situ' (situation de l'accident).

Pour les données sur l'état de la surface de la route, nous utilisons la colonne "atm" (circonstances atmosphériques) du jeu de données carac.


In [ ]:
lieux_new = lieux_new.merge(carac_new[["Num_Acc","atm"]],on="Num_Acc",how="left")

In [ ]:
lieux_new=process.clean_lieux(lieux_new)
lieux_new.drop(columns="atm",inplace=True,errors="ignore")

### Véhicules

In [ ]:
vehic_new[vehic_new["Num_Acc"].duplicated(keep=False)].sort_values(by="Num_Acc").head(20)

In [ ]:
df_obj = vehic_new.select_dtypes(['object'])
vehic_new[df_obj.columns] = df_obj.apply(lambda x: x.str.strip())
vehic_new.replace([-1,"-1",0], np.nan, inplace=True)

#### Valeurs manquantes

In [ ]:
missing_counts = vehic_new.isna().sum()
print(missing_counts[missing_counts>0],len(vehic_new))

"occutc" correspond au nombre de personnes dans le transport en commun. Elle est quasiment vide. Nous la supprimons.

Les autres colonnes ont peu de valeurs absentes. Nous les remplaçons par la valeur la plus fréquente dans les données.

In [ ]:
vehic_new = process.clean_vehicules(vehic_new)

Vérifions que la double jointure peut bien être effectuée entre usagers et vehic. Chacun des deux jeux de données possède deux colonnes d'identifiants pour la jointure : un correspondant à l'accident, un correspondant au véhicule concerné (plusieurs véhicules de plusieurs usagers différents peuvent être impliqués dans un même accident).

In [ ]:
usagers_new_2023 = usagers_new[usagers_new["year"]<2024]

In [ ]:
rows_before = usagers_new_2023.shape[0]
df_final = usagers_new_2023.merge(vehic_new, on=['Num_Acc', 'id_vehicule'], how='left')
rows_after = df_final.shape[0]
rows_after==rows_before

On peut bien effectuer la jointure sans modifier les jeux de données.

### Fusion des dataframes usagers, caractéristiques, et lieux.

Nous souhaitons fusionner les données de 2019 à 2024, pour les dataframes usagers, carac, et lieux. 
Un problème est que nous avons plusieurs lignes par accident pour "lieux". Plus précisément, nous avons une ligne par victime dans "usagers", et plusieurs lignes possibles dans "lieux" si l'accident s'est produit à une intersection. 

Comme nous observons ci-dessous, le dataframe "carac" dispose bien d'une seule ligne par accident.

In [ ]:
(carac_new["Num_Acc"].value_counts()!=1).sum()

Si nous conservons toutes les lignes de "lieux", nous devrons dupliquer les lignes des victimes, ce qui faussera nos statistiques de sécurité routière (nombre de victimes gonflé). 

Nous devons donc aplatir les lignes du dataset "lieux".

In [ ]:
for count in range(1,3):
    print(
    f"{round((lieux_new["Num_Acc"].value_counts()>count).mean(),4)*100}% des accidents \
ont strictement plus de {count} ligne(s) pour 'lieux'.")

In [ ]:
print(lieux_new.duplicated().sum())

Dans 16% des cas, nous avons deux routes pour l'accident (on est face à une intersection). Pour conserver l'information, nous créons une nouvelle variable "nb_routes" et nous sélectionnons la route principale (Autoroute>Nationale>Départementale>...) pour caractériser le lieu de l'accident. Comme les cas où on a plus de trois routes impliquées correspond à 0.2% des cas, nous pouvons supprimer ces enregistrements.

Nous faisons l'hypothèse que la gravité de l'accident sera principalement dictée par la route où la limite de vitesse est la plus élevée, où le le nombre de voies est le plus élevé, et où la largeur précise de chaque voie est la plus élevée.

On supprime d'abord les doublons parfaits.

In [ ]:
lieux_new=lieux_new.drop_duplicates()

In [ ]:
lieux_new[lieux_new.duplicated(subset=["Num_Acc"],keep=False)].head(20)
#La feature qui change peut être complètement différente d'un accident à l'autre. 
#Nous devons donc hiérarchiser.

In [ ]:
# Creation d'une feature binaire comptant le nombre de paramètres de lieux dans l'accident (1 ou 2)

lieux_new["nb_param_lieux"]=lieux_new.groupby(["year","Num_Acc"])["Num_Acc"].transform("size")
lieux_new.head(20)

In [ ]:
lieux_new = lieux_new.sort_values(
    by=[
        "year", "Num_Acc", 
        "catr",      # Critère 1 : La classe administrative (1=Autoroute gagne)
        "vma",       # Critère 2 (Si catr égal) : La vitesse (80 gagne vs 50)
        "nbv",       # Critère 3 (Si catr et vma égaux) : Le gabarit (2 voies gagne)
        "larrout"    # Critère 4 (Si tout le reste égal) : La largeur précise de la voie (nous gardons la + grande)
    ],
    ascending=[
        True, True, 
        True,        # catr : croissant (1 est mieux)
        False,       # vma : décroissant (plus grand est mieux)
        False,       # nbv : décroissant
        False        # larrout : décroissant
    ]
)
lieux_clean = lieux_new.drop_duplicates(subset=["year", "Num_Acc"], keep="first")

Maintenant que nous avons "aplati" le dataset "lieux", créons un premier dataset général qui fusionne les données de 2022,2023, et 2024 pour les dataframes **usagers**, **carac**, et **lieux** :

In [ ]:
lieux_new["Num_Acc"].isin(usagers_new["Num_Acc"]).all()
# On a bien une correspondance parfaite entre les lignes de "carac" et les lignes de "usagers". 

In [ ]:
carac_new["Num_Acc"].isin(lieux_new["Num_Acc"]).all()
# On a également une correspondance entre les lignes de "carac" et les lignes de "lieux". On peut donc bien fusionner en faisant correspondre 
# toutes les lignes avec chaque dataset

In [ ]:
KEY = ["year","Num_Acc"]

df_final2224 = usagers_new\
    .merge(lieux_clean,on=KEY,how="left")\
    .merge(carac_new,on=KEY,how="left")

# On garde toutes les lignes usagers


print(df_final2224.shape[0]==usagers_new.shape[0])

# df_final et usagers_new font bien la même longueur.

Pour l'I/O pour manipuler les données, on télécharge le dataframe en format parquet.

In [ ]:
# df_final2224.to_parquet("data_interm/df_final2224.parquet")

#### Chargement parquet

In [ ]:
# df_final2224 = pd.read_parquet("data_interm/df_final2224.parquet")

## Cartographie des accidents et analyse géographique

### Nettoyage

Nous pouvons observer la carte des accidents de la route de ce dataframe.
Avant cela, on crée un score de gravité des accidents. Les blessés ont un poids de 1, les indemnes de 0, les morts de 5. Cela ne nous servira pas pour la prédiction, mais nous sera très utile pour l'analyse exploratoire.

In [ ]:
df_final2224["grav_weight"]=df_final2224["grav_ord"].map({3:6,2:3,1:1,0:0})
s_grav_score=df_final2224.groupby("Num_Acc")["grav_weight"].transform("sum").rename("grav_score")
df_final2224["grav_score"]=s_grav_score

In [ ]:
df_map = df_final2224.drop_duplicates(subset=["Num_Acc"])[["lat","long"]].copy()

# Pour utiliser folium, on doit remplacer les virgules par des points dans long et lat

df_map["lat"]=pd.to_numeric(df_map["lat"].apply(lambda x:str(x).replace(',','.')))
df_map["long"]=pd.to_numeric(df_map["long"].apply(lambda x:str(x).replace(',','.')))

df_map.sample(20)

Pour la visualisation, on ne garde que la France Métropolitaine.

In [ ]:
df_map = df_map[
    (41<df_map["lat"])&(df_map["lat"]<52) &
    (-6<df_map["long"])&(df_map["long"]<10)
]
df_map.head(20)

In [ ]:
df_map[df_map["lat"]>90] 

Les données sont bien en GPS, pas besoin de les convertir (toutes les données sont inférieures à 90)

### Cartes, fusion avec les densités de population

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,10))
plt.scatter(df_map["long"],df_map["lat"],s=0.5,alpha=0.3)
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()

On peut clairement distinguer les zones urbaines des zones rurales. Il semble y'avoir une corrélation entre les zones à forte densité de population et le nombre d'accidents. Nous pouvons vérifier ça.

Note :  Si Folium renvoie "Make this notebook trusted to load map...", et si dans VSCode, faire Ctrl+Shift+P (Ouvre la barre de recherche avec >) puis écrire : Jupyter:Trust Notebook, puis appuyer sur Entrée. Ensuite, Restart Kernel et lancer le code. Si cela ne fonctionne pas, écrire Jupyter:Trust Notebook et relancer VSCode.

In [ ]:
import geopandas as gpd
import folium 
from folium.plugins import HeatMap

gdf = gpd.GeoDataFrame(
    df_map,
    geometry = gpd.points_from_xy(df_map.long,df_map.lat),
)
print(f"Nous affichons exactement {len(gdf)} points.")

# Carte centrée sur la France
m = folium.Map(location=[46.6,2.2],zoom_start=6,tiles="CartoDB positron") # zoom_start dézoome la carte initiale

# Nous utilisons une carte de chaleur pour mieux voir les zones dangereuses
heat_data = gdf[["lat","long"]].values.tolist()
HeatMap(heat_data,radius=10,blur=10).add_to(m)


m


Comparons les données sur les accidents aux densités de population.

In [ ]:
url_pop = "https://public.opendatasoft.com/api/explore/v2.1/catalog/datasets/correspondance-code-insee-code-postal/exports/csv?lang=fr&timezone=Europe%2FParis&use_labels=true&delimiter=%3B"
df_communes = pd.read_csv(url_pop, sep=';')

In [ ]:
# Coordonnées
if 'geo_point_2d' in df_communes.columns:
    df_communes[['lat_commune', 'lon_commune']] = df_communes['geo_point_2d'].str.split(',', expand=True).astype(float)

# Nettoyage
df_communes = df_communes.dropna(subset=['lat_commune', 'lon_commune', 'Population'])
df_communes_plot = df_communes[
    (df_communes['lat_commune'] > 41) & (df_communes['lat_commune'] < 52) &
    (df_communes['lon_commune'] > -5) & (df_communes['lon_commune'] < 10)
]

df_accidents = df_map.copy()
df_accidents['lat'] = pd.to_numeric(df_accidents['lat'].astype(str).str.replace(',', '.'), errors='coerce')
df_accidents['long'] = pd.to_numeric(df_accidents['long'].astype(str).str.replace(',', '.'), errors='coerce')

df_accidents = df_accidents.dropna(subset=['lat', 'long'])

In [ ]:
from modules import dataviz as pgf
pgf.plot_pop_nbacc(df_communes_plot,df_accidents)

La corrélation entre la densité de population d'une région et la quantité d'accidents qui y a lieu est très nette. On peut s'y attendre : plus il y'a de personnes, plus il y'a de trafic, et plus il y'a d'accidents. Par contre, nous utiliserons plus tard la densité de population pour observer si justement, elle est négativement corrélée à la gravité d'un accident dans la région ou pas.

Modifions les couleurs pour pouvoir visualiser où ont lieux les accidents les plus graves (la gravité d'un accident au score de gravité défini plus haut)

In [ ]:
# On merge sur l'index 
df_accidents_grv=pd.merge(df_accidents,s_grav_score,left_index=True,right_index=True)
df_accidents_grv=df_accidents_grv.sort_values(by="grav_score",ascending=True)
df_accidents_grv.head(20)

In [ ]:
from modules import dataviz as pgf
pgf.plot_grav_score(df_communes_plot,df_accidents_grv,
                    params={"alpha_acc":0.3,"alpha_pop":0.7}
                    )

À ce stade, nous pouvons observer la gravité moyenne des accidents hors agglomération et dans des agglomérations.

In [ ]:
sns.barplot(x=df_final2224["agg"],y=df_final2224["grav_weight"])

En effet, les accidents hors agglomération sont en moyenne bien plus grave. Cela est sans doute lié à des vitesses plus élevées et un plus fort éloignement des secours. Nous utiliserons cette feature lors du feature engineering plus bas.

### Analyse du score de gravité et de la vitesse maximale autorisée à l'échelle des communes

On souhaite maintenant observer la gravité moyenne des accidents par commune, et son rapport avec la densité de population.

On filtre les communes avec sans habitants (ou celles pour lesquelles on n'a pas l'information)

In [ ]:
df_final2224=df_final2224.rename(columns={"com":"Code INSEE"})

Pour faciliter la lecture, on convertir les unités pour la population et la superficie de chaque commune.

In [ ]:
df_communes["Population"]=df_communes["Population"]*1000 # Nombre d'habitants 
df_communes["Superficie"]=df_communes["Superficie"]/100 # Hectares -> Kilomètre carré

Créons un dataframe df_plot_densite qui contient la densité de population, le nombre d'accidents, et la gravité moyenne par commune.

In [ ]:
com_gravity = df_final2224.groupby('Code INSEE').agg(score_moyen_com=("grav_weight","mean"),
                                              nb_accidents_com=("Num_Acc","count")
).reset_index()

df_communes["densite"]=df_communes["Population"]/df_communes["Superficie"] # Milliers de personnes par hectare

df_plot_densite = com_gravity.merge(df_communes[["Code INSEE","densite"]],on="Code INSEE")
df_plot_densite=df_plot_densite[(df_plot_densite["densite"]>0)&(df_plot_densite["densite"].notna())]

On observe ci-dessous une corrélation négative entre la densité de population et la gravité des accidents.
Cela peut être lié à plusieurs facteurs : 
- En zone peu dense, les limites de vitesses ont tendance à être plus élevées. Une forte vitesse cause des accidents plus graves.
- En zone peu dense, l'intervention des pompiers et du SAMU est beaucoup plus lente. 

In [ ]:
sns.regplot(x=np.log10(df_plot_densite['densite']), y=df_plot_densite['score_moyen_com'], 
            scatter_kws={'alpha':0.3, 's': df_plot_densite['nb_accidents_com']/2},  
            line_kws={'color':'red'})

plt.title("Relation entre densité de population et sévérité des accidents")
plt.xlabel("Log(Densité de population habitant/km²)")

Nous disposons de la vitesse maximale autorisée dans la voie de circulation principale où a lieu chaque accident. Vérifions notre hypothèse, en utilisant la VMA moyenne pour chaque commune.

In [ ]:
print(f"{round(df_final2224["catr"].isin([1,2,3,4]).mean(),3)*100}% des données sur la catégorie de route sont dans 1 (Autoroute), 2 (Nationale), \
3 (Départementale), ou 4 (Communale).")

Nous analysons les outliers dans ces quatre cas.

In [ ]:
vma_analysis= process_vma.analyze_vma_outliers(df_final2224)
print(vma_analysis["nb_cas"].sum()==(df_final2224["vma"]>130).sum())

Ces quatre cas recouvrent bien la totalité des valeurs aberrantes (i.e. qui dépassent la limite française de 130 km/h)

In [ ]:
vma_analysis

Il y'a de fortes chances qu'il s'agisse simplement d'une erreur de rentrée manuelle dans le logiciel. Nous supprimons les deux enregistrements à 140 km/h et nous retirons le dernier chiffre pour les autres. Comme nous n'avons pas de données manquantes pour la catégorie de route catr, nous remplaçons les VMA manquantes (-1) par la VMA médiane pour chaque type de route.

In [ ]:
df_final2224["catr"].value_counts()

In [ ]:
df_final2224 = process_vma.clean_and_impute(df_final2224)

Nous enrichissons df_plot_densite avec les données sur la vitesse maximale autorisée.

In [ ]:
vma_com = df_final2224.groupby("Code INSEE")["vma"].mean().reset_index(name="vma_moy_com")
df_plot_densite = df_plot_densite.merge(vma_com,on="Code INSEE")

Observons le lien entre densité de population et vitesse moyenne autorisée.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

df_plot_densite['log_densite'] = np.log10(df_plot_densite['densite'])

sns.regplot(data=df_plot_densite, x='log_densite', y='vma_moy_com', 
            scatter_kws={'alpha':0.3}, line_kws={'color':'red'})

plt.title("Lien entre densité de population par commune et VMA moyenne par commune")
plt.xlabel("Densité (log10 hab/km²)")
plt.ylabel("VMA moyenne")
plt.show()

La corrélation est très claire. Maintenant, observons le lien entre VMA moyenne par commune et gravité moyenne des accidents par commune.

In [ ]:
plt.figure(figsize=(10, 6))

sns.regplot(data=df_plot_densite, x='vma_moy_com', y='score_moyen_com', 
            scatter_kws={'alpha':0.3}, line_kws={'color':'red'})

plt.title("Évolution de la gravité moyenne selon la VMA moyenne (pour chaque commune)")
plt.xlabel("Vitesse Maximale Autorisée (km/h)")
plt.ylabel("Score de Gravité Moyen")
plt.grid(True, alpha=0.3)
plt.show()

Comme attendu, la corrélation est positive. Observons la significativité de la relation.

In [ ]:
import statsmodels.api as sm
X = df_plot_densite['vma_moy_com']
y = df_plot_densite['score_moyen_com']

X = sm.add_constant(X) # Ajout de la constante, qui n'est pas présente par défaut dans les régressions linéaires de statsmodels

model = sm.OLS(y, X).fit()

print(model.summary())

La corrélation est extrêmement significative étant donné la p-valeur. Par contre le $R^2$ est très faible. La VMA n'explique qu'une partie infime de la variation de gravité moyenne des accidents entre les communes.

Cela peut s'expliquer pour plusieurs raisons. 

Il est possible que nous ayons surestimé l'influence de cette variable dans nos croyances initiales : le risque d'accident pourrait être davantage lié au comportement et aux caractéristiques de l'usager plutôt qu'aux réglementations en place. 

Il est également possible qu'il existe une relation non linéaire complexe entre la VMA et la gravité qu'un simple modèle de régression linéaire ne peut pas capturer. Nous verrons lors de la prédiction quelles variables ont l'impact le plus important sur la gravite de l'accident.


Pour la suite de nos analyses, nous ajoutons la métrique de densité de population (hab/km carré) dans notre dataset.
Attention, dans la plupart des cas nous ne disposons pas des densités de populations pour les communes de Polynésie française, de Nouvelle-Calédonie, et de Corse.

In [ ]:
df_communes["Code INSEE"] = df_communes["Code INSEE"].astype(str).str.zfill(5)
df_final2224["Code INSEE"] = df_final2224["Code INSEE"].astype(str).str.zfill(5)

In [ ]:
df_final2224=df_final2224.merge(df_communes[["Code INSEE","densite"]],on="Code INSEE",how="left")

In [ ]:
print(f"Valeurs de densité manquantes: {round(df_final2224["densite"].isna().mean()*100,3)}")

## Feature engineering 2019-2024

In [ ]:
df_cleaned1924 = df_final2224.copy()

Nous remplissons les valeurs de densité manquantes par la médiane du département, pour ne pas être trop biaisé à la hausse par les villes à forte densité de population.

Nous remplissons les valeurs restantes par la médiane des villes en agglomération ou hors agglomération.

In [ ]:
df_cleaned1924['densite'] = df_cleaned1924['densite'].replace(0, np.nan)
df_cleaned1924["densite"] = df_cleaned1924["densite"].fillna(
df_cleaned1924.groupby("dep")["densite"].transform('median')
)


Nous faisons le choix de supprimer les valeurs ne disposant pas de la densité de population  dans un premier temps, représentant moins de 1% des données. Ces valeurs correspondent aux départements d'outre-mer et à la Corse majoritairement.

In [ ]:
print(f"Représente {round(df_cleaned1924["densite"].isna().mean()*100,3)}% des données")
df_cleaned1924 = df_cleaned1924[df_cleaned1924["densite"].notna()]

Nous supprimons d'abord les colonnes de jointure (à l'exception de id_vehicule et num_veh). On conserve Num_Acc le temps du nettoyage.

In [ ]:
df_cleaned1924 = df_cleaned1924.drop(columns=["id_usager"],errors='ignore')

Nous extrayons l'heure seulement de la colonne hrmn (heures et minutes). Les minutes risquent en effet d'apporter plus de bruit que de signal.

In [ ]:
if "hrmn" in df_cleaned1924.columns:
    print(df_cleaned1924["hrmn"].isna().sum(),(df_cleaned1924["hrmn"].str.len()<5).sum())
    df_cleaned1924["hr"]=df_cleaned1924["hrmn"].str[:2].astype(int)
df_cleaned1924 = df_cleaned1924.drop(columns="hrmn",errors="ignore")

On consolide les variables de sécurité en ajoutant des variables spécifiques pour le casque, la ceinture, et l'airbag, des variables qui ont des chances d'être corrélées très négativement à la gravité des blessures. On crée aussi une variable qui correspond à la présence d'un équipement de sécurité.

In [ ]:
df_cleaned1924["secu3"].value_counts()

In [ ]:
df_cleaned1924 = engineer_features.harmonize_safety_equipment(df_cleaned1924)

Vérifions empiriquement nos intuitions.

In [ ]:
for col in ["ceinture","casque","airbag"]:
    plt.figure(figsize=(4, 3))
    sns.countplot(data=df_cleaned1924, x='ceinture', hue='grav_ord', palette='viridis')
    plt.title(f"Impact de '{col}' sur la gravité des blessures")
    plt.xlabel(f"{col} utilisé(e) (0: Non, 1: Oui)")
    plt.ylabel("Nombre d'usagers")
    plt.legend(title="Gravité")
    plt.show()


Nos intuitions sont donc bonnes, ces variables semblent avoir un fort signal pour la prédiction de la gravité des blessures.

In [ ]:
cols_corr = ['age', 'vma', 'hr', 'nbv', 'grav_weight', 'densite']
plt.figure(figsize=(10, 8))
sns.heatmap(df_cleaned1924[cols_corr].corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Heatmap des corrélations numériques")
plt.show()

On crée plusieurs nouvelles features à partir des corrélations observées. En particulier, un ratio vma/densité (plus il est élevé, et plus la VMA est élevée par rapport à la densité de la région, ce qui est particulièrement dangereux (moins d'infrastructures en cas d'accidents, faible proximité des soins)). On utilise la log densité, pour éviter d'avoir une queue trop longue vers les fortes densités, moins fréquentes.

On crée aussi *vma/nbv*, qui correspond à la vitesse maximale autorisée divisée par le nombre de voies sur la route. Pour cela spécifiquement, on remplace les zéros (non renseigné) par la valeur la plus représentée dans les données nbv, soit les routes à deux voies.

In [ ]:
df_cleaned1924["densite_log"] = np.log1p(df_cleaned1924["densite"])

In [ ]:
df_cleaned1924["vma_densite_ratio"] = df_cleaned1924["vma"] / df_cleaned1924["densite_log"]
df_cleaned1924['vma_nbv_ratio'] = df_cleaned1924['vma'] / df_cleaned1924['nbv'].replace(0, 2)

On utilise maintenant les variables liées au lieu de l'accident. Est-ce que la "complexité" du lieu affecte la gravité de l'accident ? Nous observons le lien entre la gravité de l'accident, la confguration de la route, et le type de collision.

In [ ]:
dic0 = {"plan":"Tracé en plan",
        "prof":"Déclivité de la route",
        "int":"Type d'intersection",
        "vosp":"Accident sur une voie réservéee",
        }
dataviz.plot_dic_grav(df_cleaned1924,dic0)

La gravité pondérée dépasse 1 lorsqu'il n'y a pas d'intersection (int=1) et lors des passages à niveau (int=8). Les accidents aux passages à niveau peuvent être des véhicules percutés par des trains, ce qui explique leur forte gravité.

La gravité est croissante avec la déclivité de la route et le type de tracé (1 étant rectiligne, 2 en courbe à gauche, 3 en courbe à droite, 4 en S).

Enfin, l'accident a tendance à être moins grave lorsqu'il a lieu sur une voie réservée (sans doute car les règlementations de vitesse sont plus strictes et les usagers sont plus attentifs). Le One-Hot Encoding créera une variable binaire pour prendre en compte cet effet.

Nous créons une nouvelle variable d'interaction *route_danger* (déclivité > 1 et tracé en plan > 1).

In [ ]:
df_cleaned1924 = engineer_features.trace_route(df_cleaned1924)

In [ ]:
sns.barplot(x=df_cleaned1924["route_danger"],y=df_cleaned1924["grav_weight"])

Cette variable a un fort pouvoir explicatif, avec un faible écart-type sur chaque moyenne des gravités.

Concentrons nous maintenant sur les usagers et spécifiquement sur la variable âge.

In [ ]:
gravity_age = df_cleaned1924.groupby("age")["grav_weight"].mean().reset_index()
plt.figure(figsize=(12,6))
sns.lineplot(data=gravity_age,x="age",y="grav_weight",color="red",linewidth=2)
plt.title("Gravité moyenne de l'accident en fonction de l'âge",fontsize=14)
plt.xlabel("Âge de l'usager (années)")
plt.ylabel("Score de gravité moyen")

La distribution des âges des usagers est très inégale, d'où une courbe peu lisse. La forme de la courbe nous incite à utiliser des modèles de prédiction qui gèrent les relations non linéaires.

La forme de la courbe est attendue : la gravité moyenne des blessures d'un individu est importante lorsqu'il est jeune et lorsqu'il est vieux (car sa constitution est plus fragile), et elle est plus faible lorsqu'il est d'âge moyen. On peut aussi imaginer que le pic de gravité observé en fin d'adolescence est lié à une conduite peu responsable ou festive à cet âge.

In [ ]:
sns.kdeplot(df_cleaned1924["age"])

On utilise des tranches d'âge pour pallier le problème d'importantes inégalités de distribution dans les données.

In [ ]:
df_cleaned1924["age_bin"] = pd.cut(df_cleaned1924["age"],bins=range(0,101,5)) # De 0 à 100 par incréments de 5

Etant donné l'importance de la variable âge dans la prédiction, nous supprimons les lignes où cette valeur est nulle. Ces lignes représentent 0.15% du jeu de données.

In [ ]:
print(f"{round(df_cleaned1924["age_bin"].isna().mean()*100,3)}%")

In [ ]:
df_cleaned1924 = df_cleaned1924[~df_cleaned1924["age_bin"].isna()]

In [ ]:
sns.countplot(df_cleaned1924["age_bin"])

In [ ]:
age_bin_gravity = df_cleaned1924.groupby("age_bin")["grav_weight"].mean().reset_index()

plt.figure(figsize=(10,6))
sns.barplot(data=age_bin_gravity,x="age_bin",y="grav_weight",color="salmon")
plt.title("Gravité moyenne de l'accident pour l'usager par tranche d'âge")
plt.xticks(rotation=45)
plt.xlabel("Tranches d'âge, incréments de 5 ans")
plt.ylabel("Gravité de l'accident")

On observe la même tendance, avec une première augmentation de la gravité des accidents lors des jeunes âges, puis une diminution jusqu'à la trentaine. Enfin, la gravité moyenne des accidents monte nettement à partir de la fin de la quarantaine, de plus en plus.

Par contre, on aurait pu s'attendre à une plus forte gravité moyenne des accidents pour les plus jeunes (notamment entre 0 et 5 ans). Les facteurs de sécurité routière (notamment la présence d'équipements) doit beaucoup jouer.

In [ ]:
plt.figure(figsize=(12, 7))

df_cleaned1924["Protection"] = df_cleaned1924["safety"].map({1 : 'Avec équipement', 0 : 'Sans équipement'})

sns.barplot(data=df_cleaned1924, x='age_bin', y='grav_weight', hue='Protection', 
             palette={'Avec équipement': 'green', 'Sans équipement': 'red'},
             linewidth=2.5)
plt.title("Gravité moyenne de l'accident pour l'usager par tranche d'âge et selon la présence de protection")
plt.xticks(rotation=45)
plt.xlabel("Tranches d'âge, incréments de 5 ans")
plt.ylabel("Gravité de l'accident")

On voit bien qu'en prenant en compte la présence d'un équipement de sécurité, les tendances sont très différentes. En particulier, la gravité de l'accident est deux fois plus faible pour les usagers de 0 à 5 ans. Cela est lié à l'utilisation de dispositif enfants, très sécuritaires.

On crée une nouvelle variable qui correspond à la fragilité physiologique. On considère les enfants (jusqu'à 12 ans) et les personnes agées à partir de 65 ans (ce second seuil est arbitraire).

In [ ]:
df_cleaned1924["est_fragile"] = ((df_cleaned1924["age"]<12)|(df_cleaned1924["age"]>70)).astype(int)

Nous nous concentrons maintenant sur trois caractéristiques spécifiques aux usagers : la nature du trajet, la localisation des piétons, et la catégorie d'usager.

In [ ]:
dic2 = {"trajet":"Nature du trajet",
        "catu":"Catégorie d'usager",
        "locp":"Localisation du piéton",
        "etatp":"Si le piéton accidenté était seul",
        "actp":"Action du piéton au moment de l'accident"}
dataviz.plot_dic_grav(df_cleaned1924,dic2)

On voit que les types de trajets 3 et 5 (respectivement courses et loisirs) présentent un risque substantiellement plus important. Le trajet 4 (trajet professionnel) présente un faible risque de gravité. Une nouvelle colonne correspondant aux trajets professionnels sera créée lors du One-Hot Encoding que nous effectuerons pour la prédiction. On crée une nouvelle colonne *trajet_haute_grav*. On pouvait s'y attendre : les trajets de loisir incluent des conduites moins attentives, et donc plus dangereuses. 

On voit également que la gravité des accidents est bien plus forte pour les piétons, ce qui est attendu puisqu'ils ne sont pas protégés. Elle sera créée par le One-Hot Encoding.

On voit que les piétons seuls sont plus à risque.

Enfin, l'action du piéton au moment de l'accident semble influencer sa gravité. En particulier, les piétons avec animal (6) sont le plus à risque, malgré une certaine variance dans la donnée. On crée une nouvelle variable qui considère les piétons seuls et avec animal.

Un piéton connaît en moyenne un plus grave accident s'il est à plus de 50 mètres d'un passage piéton ou s'il est sur l'accotement, et s'il est seul. Nous créons une variable *ped_loc_seul* qui prend en compte sa localisation et s'il était seul ou pas.


In [ ]:
df_cleaned1924 = engineer_features.pietons_trajet(df_cleaned1924)

Visualisons ces nouvelles variables :

In [ ]:
dic_ = {"seul_avec_animal":"Seul avec animal","ped_loc_seul":"à plus de 50m d'un passage piéton ou sur l'accotement, et seul"}
dataviz.plot_dic_grav(df_cleaned1924,dic_)

Ces deux variables semblent avoir un fort pouvoir prédictif, ce qui est prometteur et facilitera le travail de l'algorithme de ML plus bas.

Revenons aux circonstances générales de l'accident. Parmi les autres variables présentes dans nos données, nous avons la luminosité de la route, l'heure, et l'état de la route, qui peuvent jouer un grand rôle dans la gravité de l'accident.

In [ ]:
dic1 = {"lum":"Conditions d'éclairage",
        "atm":"Conditions atmosphériques",
        "surf":"Etat de la surface de la route",
        "hr":"Heure de l'accident",
        }
dataviz.plot_dic_grav(df_cleaned1924,dic1)

On peut clairement distinguer des tendances. Les accidents sont en moyenne plus grave lorsque la route n'est PAS éclairée (lum vaut 3 ou 4 d'après la description des données), avec un écart-type faible. 

Lorsqu'on a du brouillard (atm = 5) ou des tempêtes (atm = 6), on a également une gravité moyenne plus élevée des accidents, avec un écart-type faible encore une fois.

L'état de la surface est particulièrement liée à la gravité des accidents en très mauvaises conditions : 4-Inondée, 6-Boue.
Notons que 9-Autre est aussi importante dans la relation entre surf et grav_weight et entre lum et grav_weight.

Enfin, on observe clairement que les accidents sont plus graves dans la soirée et dans la nuit. On conserve les heures, en les transformant de façon cyclique (pour que les heures soient projetées comme des coordonnées sur un cercle), et on crée une variable soiree.

In [ ]:
df_cleaned1924 = engineer_features.lum_atm_surf_hr(df_cleaned1924)

Enfin, on lie les variables agglomération et densité. Pour pouvoir les comparer, on discrétise la variable de densité dans un premier temps. Pour des raisons de clarté visuelle, on map le type de zone en Hors-Agglo et En-Agglo.

In [ ]:
df_cleaned1924['densite_tranche'] = pd.qcut(df_cleaned1924['densite'], q=4, 
                                            labels=['Faible', 'Moyenne', 'Forte', 'Très Forte'])

df_cleaned1924['zone_type'] = df_cleaned1924['agg'].map({1: 'Hors-Agglo', 2: 'En-Agglo'})

In [ ]:
plt.figure(figsize=(12, 6))
sns.pointplot(data=df_cleaned1924, x='densite_tranche', y='grav_weight', 
              hue='zone_type', dodge=True, capsize=.1)

plt.title("Impact de la localisation (agg) sur la gravité à densité égale")
plt.ylabel("Gravité Moyenne (Score Physique)")
plt.xlabel("Niveau de densité de population")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

Il est très clair que pour des niveaux de densité de population faibles à forts (trois premiers quartiles), les accidents sont plus graves hors agglomération. Les routes hors agglomération à faible densité sont souvent des routes départementales ou nationales à vitesse élevée, et avec peu d'infrastructures à proximité. Pour capturer cet effet directement, on crée une variable *rural_isolé*  qui donne l'information sur les zones hors agglomération à faible densité.

In [ ]:
df_cleaned1924['rural_isole'] = ((df_cleaned1924['agg'] == 1) & (df_cleaned1924['densite'] < df_cleaned1924['densite'].median())).astype(int)

In [ ]:
sns.barplot(x=df_cleaned1924['rural_isole'],y=df_cleaned1924["grav_weight"])

Le signal de cette variable est extrêmement fort ; les moyennes sont très disparates mais l'écart-type des valeurs dans chaque catégorie est très faible.

L'étape de feature engineering est quasi-complète : nous observons maintenant le dataset vehic_new, à partir duquel on va pouvoir créer une variable d'estimation de l'énergie cinétique appliquée lors de l'accident, ce qui a des chances d'être déterminant pour prédir sa gravité.

Plusieurs variables semblent intéressantes ici. La catégorie de véhicule (liée à ses dimensions et à son poids), le type d'obstacle heurté (si on en a heurté un), le type d'obstacle MOBILE heurté s'il y'en a eu un, le point de choc initial, la manoeuvre principale avant l'accident, et le type de motorisation du véhicule.

In [ ]:
df_cleaned23v = df_cleaned1924[df_cleaned1924["year"]<=2023].copy()
vehic_new = vehic_new.drop(columns=["year","num_veh"])
df_cleaned23v = df_cleaned23v.merge(vehic_new,on=["Num_Acc","id_vehicule"],how="left")

In [ ]:
dic3 = {
    "catv": "Catégorie du véhicule impliqué (voiture, deux-roues, poids lourd...)",
    "obs": "Nature de l'obstacle fixe heurté lors de l'accident (arbre, mur, poteau...)", 
    "obsm": "Nature de l'obstacle mobile heurté (autre véhicule, piéton, animal...)", 
    "choc": "Localisation du point de choc initial sur le véhicule (avant, arrière, côté...)",
    "manv": "Manoeuvre principale effectuée par le véhicule avant l'accident (dépassement, tournant...)",
    "motor": "Type de motorisation du véhicule (hydrocarbures, hybride, électrique...)",
}

dataviz.plot_dic_grav(df_cleaned23v,dic3)

**Catégorie du véhicule** : On voit que les usagers de véhicules lourds s'en sortent beaucoup mieux.

**Nature de l'obstacle fixe heurté** : Particulièrement élevé pour les têtes d'aqueduc (17), arbre (2)... La gravité augmente avec la résistance de l'obstacle fixe heurté.

**Nature de l'obstacle mobile heurté** : Gravité pondérée moyenne particulièrement élevée pour les chocs avec un véhicule sur rails (4), un animal domestique (5) (probablement des animaux d'élevage de taille importante, comme des boeufs), et avec un animal sauvage (6) (sans doute des animaux imposants, comme des cervidés).

**Localisation du point de choc initial** : La gravité est la plus élevée pour les chocs frontaux (1) et latéraux (7 et 8), et encore plus lorsqu'il y a eu plusieurs chocs simultanément (9).

**Manoeuvre effectuée par le véhicule avant l'accident** : On observe une forte variance selon la manoeuvre effectuée. La gravité est particulièrement importante pour les véhicules à contresens (5) et franchissant le terre-plein central (6), les véhicules dépassant (à gauche 17, à droite 18), et pour les véhicules qui quittent leurs trajectoires initiales de manière latérale (13 et 14). 13 et 14 suggèrent un écart brusque ou une perte de contrôle imprévue.

**Type de motorisation du véhicule** : Enfin, les piétons (5) sont naturellement beaucoup plus à risque, n'étant a priori pas protégés. Les véhicules hybrides électriques et à hydrogène (4) sont aussi à moindre risque. Cela peut s'expliquer par plusieurs raisons.

Ces véhicules sont plus récents donc potentiellement mieux équipés en sécurité passive. Ils ont également tendance à être plus lourds (batteries lourdes), donc encaissent mieux les chocs. Ils sont également plus utilisés en milieu urbain, où on a vu que les accidents ont tendance à être moins graves. Aussi, ces véhicules sont plus chers, donc peuvent inciter les conducteurs à adopter une conduite moins risquée. Enfin, ces véhicules représentent un faible échantillon du total, ce qui peut biaiser le calcul.

- - - - - 
Nous créons plusieurs nouvelles cross-features à partir de ces observations et en les combinant aux autres variables. 

Nous attribuons à chaque type de véhicule un coefficient de masse, un proxy pour sa résistance (qui le protège mais peut aggraver la situation des usagers des autres véhicules). On crée ainsi une variable *is_vulnerable* et une variable *is_heavy*. En utilisant ce coefficient, nous créons une variable d'énergie cinétique en multipliant ce proxy par la VMA au carré, *kinetic_energy_score*.

*vulnerable_outside_agg* capture le risque combiné de la vulnérabilité et de la circulation hors agglomération, structurellement plus risquée pour des raisons mentionnées plus haut.

*extreme_fixed_obs_impact* isole les chocs contre les obstacles les plus rigides.

*curve_drift_risk* isole les déports survenant en courbe ; ceux-ci ont des chances d'être plus risqués, puisqu'ils peuvent être un proxy d'une conduite inadaptée à la géométrie de la chaussée.

*side_impact_junction* lie les chocs latéraux aux collisions en intersection, où ce genre de chocs est plus violent.

*night_fixed_obs* lie la conduite nocturne aux chocs avec des obstacles dangereux, où le choc est pris de plein fouet.

*heavy_vs_vulnerable_collision* prend en compte les collisions entre véhicules massifs et véhicules vulnérables. Ce scénario présente sans doute un taux de gravité pondérée moyen particulièrement élevé.



In [ ]:
df_cleaned23v = engineer_features.vehic_features(df_cleaned23v)

# Données 2005-2021

Nous téléchargeons les données de 2005-2021 disponibles en ligne, et ne conservons que les données de 2005 à 2018 inclus. 

## Chargement des données

In [ ]:
url_05_21 = {
    "carac" : "https://www.data.gouv.fr/api/1/datasets/r/a3cac8bc-4a07-4124-8a08-633a3a91d40b",
    "lieux" : "https://www.data.gouv.fr/api/1/datasets/r/b7f25e45-de32-4801-b0eb-62989f1a7406",
    "usagers" : "https://www.data.gouv.fr/api/1/datasets/r/a64b1b9f-4d56-4b26-ae90-9f40b878e109",
    "vehic" : "https://www.data.gouv.fr/api/1/datasets/r/41aad2f4-5f76-4196-836e-51cac17dba51"
}

In [ ]:
df_0521={}
for type,link in url_05_21.items():
    df_0521[type] = pd.read_csv(link,
                               encoding="latin-1",  # Old files, with a different encoding than the recent ones
                               sep=",")

# Prend environ 3 min à tourner

In [ ]:
print(df_0521["carac"].columns,"\n",carac_new.columns)

In [ ]:
print(df_0521["usagers"].columns,"\n",usagers_new.columns)

Comme nous avons déjà mené l'analyse sur les jeux de données datant d'après 2019 inclus, nous supprimons ces années. 
Dans un premier temps, nous renommons la colonne année et la colonne d'identifiant des accidents, puis nous vérifions que tous les accidents comportent l'année.
Nous supprimons également la colonne d'index redondante Unnamed : 0 qui s'est glissée dans le jeu de donneés.

In [ ]:
for type in ["usagers","carac","lieux","vehic"]:
    df_0521[type] = df_0521[type].rename(columns={"annee":"year","num_acc":"Num_Acc"})
    print(df_0521[type]["year"].isna().sum())
    if "Unnamed: 0" in df_0521[type].columns:
        df_0521[type].drop(columns="Unnamed: 0",inplace=True)

Tous les accidents comportent bien leur année d'occurrence. On peut donc procéder à la réduction du jeu de données.

In [ ]:
for type in ["usagers","carac","lieux","vehic"]:
    df_0521[type] = df_0521[type][df_0521[type]["year"]<2019]

Ici aussi, nous avons bien un seul enregistrement des caractéristiques pour chaque identifiant d'accident. Nous avons par ailleurs une colonne d'identifiants Num_Acc identiques pour *carac* et pour *lieux*, ce qui va faciliter la fusion des jeux de données.

In [ ]:
print((df_0521["carac"]["Num_Acc"].value_counts()>1).sum())

In [ ]:
(df_0521["carac"]["Num_Acc"]!=df_0521["lieux"]["Num_Acc"]).sum()

Par ailleurs, tous les accidents recensés dans *carac* (et donc dans *lieux*) le sont aussi dans *usagers*, et réciproquement.

In [ ]:
(df_0521["carac"]["Num_Acc"]!=(df_0521["usagers"]["Num_Acc"].unique())).sum()

In [ ]:
# for name in names:
#     df_0521[name].to_csv(f"data_interm/old/{name}_old.csv",index=False)

carac_old = df_0521["carac"]
usagers_old = df_0521["usagers"]
lieux_old = df_0521["lieux"]
vehic_old = df_0521["vehic"]

### Chargement des données en .csv

In [ ]:
# carac_old = pd.read_csv("data_interm/old/carac_old.csv")
# usagers_old = pd.read_csv("data_interm/old/usagers_old.csv")
# lieux_old = pd.read_csv("data_interm/old/lieux_old.csv")
# vehic_old = pd.read_csv("data_interm/old/vehic_old.csv")

On supprime directement les colonnes qui sont intégralement vides, artéfacts des jeux de données post-2019.

In [ ]:
carac_old = carac_old.dropna(axis=1, how="all")
usagers_old = usagers_old.dropna(axis=1, how="all")
lieux_old = lieux_old.dropna(axis=1, how="all")
vehic_old = vehic_old.dropna(axis=1,how="all")

On applique directement nos codes de nettoyage précédents. 

In [ ]:
rows_before = [len(carac_old),len(usagers_old),len(lieux_old),len(vehic_old)]

In [ ]:
carac_old = process.clean_carac(carac_old)
usagers_old = process.clean_usagers(usagers_old)
lieux_old = process.clean_lieux(lieux_old)
vehic_old = process.clean_vehicules(vehic_old)

In [ ]:
print(rows_before,f"\n{len(carac_old),len(usagers_old),len(lieux_old),len(vehic_old)}")

In [ ]:
KEY = ["Num_Acc","year"]
carac_lieux = carac_old.merge(lieux_old, on=KEY, how="inner")
carac_lieux_usagers = usagers_old.merge(carac_lieux, on=KEY, how="left",validate="many_to_one") 

On peut bien effectuer la jointure sur Num_Acc et num_veh sans modification dans le nombre de lignes.

In [ ]:
rows_before = carac_lieux_usagers.shape[0]
df_final = carac_lieux_usagers.merge(vehic_old, on=['Num_Acc', 'num_veh'], how='left')
rows_after = df_final.shape[0]
rows_after==rows_before

In [ ]:
df_cleaned_old = carac_lieux_usagers.merge(vehic_old.drop(columns="year"),on=["Num_Acc","num_veh"],how="left")

Les données de latitude et de longitude sont massivement indisponibles.

In [ ]:
df_cleaned_old[["lat","long"]].isna().sum()

Comme l'analyse géographique n'apportera pas forcément davantage que celle qui a été menée plus haut sur les données de 2019 à 2024, nous supprimons ces deux colonnes. 

Nous pouvons voir ci-dessous que nous avons l'information sur quasiment l'intégralité des communes où ont eu lieu les accidents. Nous supprimons les 6 enregistrements (chiffre infinitésimal eu égard à la longueur du jeu de données).

In [ ]:
df_cleaned_old[["dep","com"]].isna().sum()

In [ ]:
df_cleaned_old = df_cleaned_old.dropna(subset=["com"])
df_cleaned_old = df_cleaned_old.drop(columns=["long","lat"])

## Nettoyage des données 2005-2018

Steps suivants : 

Observons les différences dans les noms de colonnes du format des anciens datasets (2005-2021), et des nouveaux datasets (2022-2024).

In [ ]:
cols_recentes = set(df_cleaned23v.columns)
cols_anciennes = set(df_cleaned_old.columns)

unique_recent = cols_recentes-cols_anciennes # Soustraire les sets permet de ne garder que les colonnes uniquement présentes en 2022-2024
unique_ancien = cols_anciennes-cols_recentes # Colonnes uniquement présentes en 2005-2021

print(unique_ancien,unique_recent)
# print(cols_anciennes,cols_recentes)

Nous supprimons l'ancienne colonne GPS (qui ne prend qu'une seule mystérieuse valeur, "M"), inutile.

"env1" correspond à la proximité d'une école. Nous la conservons pour les analyses de ce jeu de données, mais elle n'est plus collectée après 2019.

La colonne "com" était présente dans notre ancien dataset. Nous l'avons remplacée par le Code INSEE de la commune. Nous allons faire la même chose pour ce jeu de données.

"hrmn" avait été supprimée pour projeter les heures sur un cercle et supprimer l'effet de "saut" entre 23h00 et minuit. Nous allons effectuer la même transformation.

Enfin, la colonne "secu" correspond de 2005 à 2021 à un code sur deux caractères : 

Le premier concerne l'existence d'un équipement de sécurité:
    
    1 – Ceinture,
    2 – Casque,
    3 – Dispositif enfants,
    4 – Equipement réfléchissant,
    9 – Autre

Le second concerne l'utilisation de cet équipement de sécurité :
    
    1 – Oui,
    2 – Non,
    3 – Non déterminable

Dans les versions plus récentes, il est question de l'existence ET de l'utilisation d'un équipement de sécurité, jusqu'à trois à la fois (secu1,secu2,secu3):
    
    -1 – Non renseigné  
    0 – Aucun équipement  
    1 – Ceinture  
    2 – Casque  
    3 – Dispositif enfants  
    4 – Gilet réfléchissant  
    5 – Airbag (2RM/3RM)  
    6 – Gants (2RM/3RM)  
    7 – Gants + Airbag (2RM/3RM)  
    8 – Non déterminable  
    9 – Autre

Cette nouvelle nomenclature (secu1,secu2,secu3) qui permet une analyse plus granulaire a été introduite en 2019. Nous allons créer

Dans les étapes suivantes, nous pouvons donc abandonner "gps", "id_vehicule", "id_usager", et réfléchir à une modification de la colonne "secu" pour harmoniser les données 05-21 avec 22-24.


Il est capital de noter que la nomenclature des blessés a changé après 2019 : blessés hospitalisés et blessés léger ne sont plus définis de la même manière. La documentation est floue sur les détails de la modification du label.

In [ ]:
df_cleaned_old.drop(columns=["gps"],inplace=True,errors="ignore")
df_cleaned_old["nb_param_lieux"] = 1

### Valeurs manquantes

On gère d'abord les valeurs manquantes non nettoyées par les codes précédents.

In [ ]:
missing_counts = df_cleaned_old.isna().sum()
missing_counts[missing_counts>0]

Nous supprimons directement les enregistrements absents de catr (seulement deux lignes). Concernant env1, nous n'avons pas de détails sur les valeurs dans la description des données.

In [ ]:
df_cleaned_old = df_cleaned_old.dropna(subset="catr")

In [ ]:
sns.barplot(x=df_cleaned_old["env1"],y=df_cleaned_old["grav_ord"])

Les effets de 0 et de 99 étant très similaires (moyenne très proche, variance très faible), sans information supplémentaire, nous les fusionnons.

In [ ]:
mapping = {
    3.0: 'ecole_faible_danger',
    0.0: 'standard_ou_inconnu',
    99.0: 'standard_ou_inconnu'
}

df_cleaned_old['env1'] = df_cleaned_old['env1'].map(mapping).fillna('standard_ou_inconnu')

Enfin, nous transformons les valeurs manquantes de secu en "0", qui semble être la valeur par défaut affectée aux valeurs manquantes, bien que ça ne soit pas précisé dans le document de description. En outre, certaines valeurs n'ont aucun sens : 1, 2, ou 3 en particulier.

In [ ]:
df_cleaned_old["secu"].value_counts()

In [ ]:
df_cleaned_old["secu"] = df_cleaned_old["secu"].fillna(0)

In [ ]:
# df_cleaned_old.to_parquet("data_interm/df_cleaned_old.parquet")

### Chargement parquet

In [ ]:
# df_cleaned_old = pd.read_parquet("data_interm/df_cleaned_old.parquet")

## Feature engineering 2005-2018

Notre code harmonise les deux époques en extrayant deux indicateurs binaires fondamentaux : ceinture et casque.

In [ ]:
from modules import engineer_features

df_cleaned_old = engineer_features.harmonize_safety_equipment(df_cleaned_old)

# Vérification rapide
print("Répartition Ceinture (Avant 2019) :")
print(df_cleaned_old['ceinture'].value_counts())

print("\nRépartition Casque (Avant 2019) :")
print(df_cleaned_old['casque'].value_counts())

In [ ]:
sns.countplot(data=df_cleaned_old,x="ceinture",hue="grav_ord")

La définition de "Blessé hospitalisé" a changé depuis 2018, ce qui rend la comparaison exacte des variables 2 et 3 impossible  2005-2018 et 2019-2024. On comparera simplement l'efficacité de la prédiction 2005-2018 à la prédiction 2019-2024.

Nous appliquons nos codes de feature engineering.

In [ ]:
df_cleaned_old = engineer_features.lum_atm_surf_hr(df_cleaned_old)
df_cleaned_old = engineer_features.pietons_trajet(df_cleaned_old)
df_cleaned_old = engineer_features.trace_route(df_cleaned_old)
df_cleaned_old = engineer_features.vehic_features(df_cleaned_old)

Après l'usage des codes, il demeure 22 valeurs manquantes pour sur plusieurs millions d'enregistrements, pour quelques variables du jeu de données véhic (valeurs manquantes liées à l'ajout de variables post-2019, qui ne sont pas présentes dans cet ancien jeu de données). Nous les supprimons.

In [ ]:
missing = df_cleaned_old.isna().sum()
missing[missing>0]

In [ ]:
df_cleaned_old = df_cleaned_old.dropna()

Préparons maintenant les données pour la prédiction.

La VMA n'est pas disponible dans ce jeu de données. La variable "motor" non plus. Nous recréons les variables densite_log, rural_isole, age_bin, et est_fragile, créées dans la partie *Feature engineering 2019-2024*.

In [ ]:
df_cleaned_old["age_bin"] = pd.cut(df_cleaned_old["age"],bins=range(0,101,5)) # De 0 à 100 par incréments de 5
print(f"{round(df_cleaned_old["age_bin"].isna().mean()*100,2)}%")

Nous supprimons les enregistrements hors de cette gamme d'âges, un pourcentage très faible des données.

In [ ]:
df_cleaned_old = df_cleaned_old.dropna(subset="age_bin")

In [ ]:
df_cleaned_old["est_fragile"] = ((df_cleaned_old["age"]<12)|(df_cleaned_old["age"]>70)).astype(int)

In [ ]:
df_cleaned_old['dep_clean'] = (df_cleaned_old['dep'].astype(int)// 10).astype(str).str.zfill(2)
df_cleaned_old['com_clean'] = df_cleaned_old['com'].astype(int).astype(str).str.zfill(3)
df_cleaned_old["Code INSEE"] = df_cleaned_old['dep_clean'] + df_cleaned_old['com_clean']

In [ ]:
if 'densite' not in df_cleaned_old.columns:
    df_cleaned_old=df_cleaned_old.merge(df_communes[["Code INSEE","densite"]],on="Code INSEE",how="left")

On remplace les valeurs manquantes par la médiane de densité du département.

In [ ]:
df_cleaned_old['densite'] = df_cleaned_old['densite'].replace(0, np.nan)
df_cleaned_old["densite"] = df_cleaned_old["densite"].fillna(
df_cleaned_old.groupby("dep_clean")["densite"].transform('median')
)

On supprime les données restantes (outre-mer, Corse) pour ne pas fausser le modèle, pour lequel la densité de population est une variable importante (a priori, et a posteriori d'après notre modèle entraîné sur les données de 2019 à 2024.)

In [ ]:
print(f"Représente {round(df_cleaned_old["densite"].isna().mean()*100,3)}% des données")

In [ ]:
df_cleaned_old = df_cleaned_old.dropna(subset="densite")

In [ ]:
df_cleaned_old["densite_log"] = np.log1p(df_cleaned_old["densite"])

In [ ]:
df_cleaned_old['rural_isole'] = ((df_cleaned_old['agg'] == 1) & 
                                 (df_cleaned_old['densite'] < df_cleaned_old['densite'].median())
                                 ).astype(int)

# Prédictions

## Prédiction 2019-2024

Nous utilisons pour notre prédiction un algorithme de forêts aléatoires. Le bagging appliqué aux arbres de décisions permet de réduire la variance de nos prédictions et ainsi de réduire les risques de surentraîner le modèle.

Ces modèles gèrent nativement la non-linéarité, ce qui est essentiel dans le cas de notre jeu de données, qui comporte des dizaines de variables ayant des effets complexes sur la gravité de l'accident. Par ailleurs, les forêts aléatoires capturent mieux les effets de seuil que des régressions logistiques, puisqu'elles procèdent par paliers, et sont robustes aux valeurs extrêmes.

Ils capturent mieux les interactions complexes entre variables, notamment celles que nous n'aurions pas remarquées pendant la phase de feature engineering. Ces modèles sont aussi particulièrement robustes aux variables catégorielles (la majeure partie de nos variable) en supportant une quantité massive de colonnes résultant du One-Hot Encoding. 

Enfin, grâce au suivi de la diminution de l'impureté de Gini pendant la construction des arbres, le modèle fournit un classement des variables les plus discriminantes, ce qui permet de valider physiquement nos résultats.

### Analyse sans véhicules

In [ ]:
categorical_features_eng = [
    "place", "catu", "sexe", "trajet", "locp", "actp", "etatp", "age_bin", 
    "catr", "circ", "vosp", "prof", "plan", "larrout", "surf", "infra", 
    "situ", "mois", "jour", "lum", "dep", "agg", "int", "atm", "col",
]

numerical_features_eng = [
    'vma', 'nbv', 'nb_param_lieux', 'densite_log', 'hour_sin', 'hour_cos', 
    'vma_densite_ratio', 'vma_nbv_ratio', 'ceinture', 'casque', 'airbag', 
    'safety', 'route_danger', 'int_danger', 'est_fragile', 'trajet_haute_grav', 
    'seul_avec_animal', 'surface_critique', 'meteo_critique', 'anomalie_atm', 
    'anomalie_surf', 'not_lum', 'rural_isole', 'ped_loc_seul',
]
# numerical_features_eng inclut les variables binaires

In [ ]:
from sklearn.model_selection import train_test_split
from modules.prediction import get_pipeline, run_cross_val, evaluate_predictions, get_feature_importance_df

Nous utilisons un Random Forest avec poids équilibrés (class_weight="balanced"), car les classes Tués et blessés hospitalisés sont particulièrement minoritaires dans les données.

Pour des raisons de coût computationnel, on n'effectue pas de GridSearch ou de RandomSearch et on choisit des valeurs par défaut de 100 arbres de décisions et une profondeur maximale de 20 par arbre (distance maximale entre la racine et les feuilles). 

In [ ]:
X = df_cleaned1924[numerical_features_eng + categorical_features_eng]
y = df_cleaned1924['grav_ord']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

 # n_jobs est à fixer en fonction du nombre de coeurs du CPU qu'on souhaite utiliser. Par défaut, ils sont tous utilisés.
pipeline = get_pipeline(categorical_features_eng,n_jobs=15)

Observons les résultats obtenus par la cross-validation. Dans notre cas, le f1-score est plus pertinent que l'accuracy. En pratique, il est moins problématique de classifier les indemnes en blessés/tués que de classifier les blessés/tués en indemne. Nous nous concentrons sur le recall et la précision, et souhaitons maximiser la moyenne harmonique des deux (le f1-score).

En raison d'un temps de calcul particulièrement long, nous commentons ici la cross-validation, et laissons sa reproduction à la discrétion du lecteur. Ses résultats donnent une variance très faible et un f1-score moyen identique (au centième près) à celui obtenu par le modèle plus bas.

In [ ]:
# run_cross_val(pipeline,X_train,y_train,n_jobs=15) 

Les résultats de cross-validation, comparés au modèle entraîné plus bas, permettent de valider que nous ne surentraînons pas le modèle.

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
labels = [
    '0-Indemne', 
    '1-Léger', 
    '2-Hospitalisé', 
    '3-Tué'
]

cm = evaluate_predictions(pipeline,X_test,y_test,labels)

In [ ]:
from sklearn.metrics import classification_report
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(cmap='viridis', ax=ax, values_format='.2f')

plt.title("Matrice de Confusion Normalisée\n(Capacité de détection par classe de gravité)")
plt.show()

In [ ]:
feature_importance_df = prediction.get_feature_importance_df(pipeline,categorical_features_eng,numerical_features_eng)

plt.figure(figsize=(12, 10))
sns.barplot(data=feature_importance_df.head(20), x='Importance', y='Feature', palette='magma')
plt.title("Top 20 des variables les plus déterminantes pour la gravité")
plt.xlabel("Importance (Gini Importance)")
plt.ylabel("Variables")
plt.tight_layout()
plt.show()

In [ ]:
hospit_as_indemne = cm[2, 0]
total_hospit = cm[2, :].sum()

print(f"ERREUR CRITIQUE : {hospit_as_indemne} blessés hospitalisés ont été prédits 'Indemnes'.")
print(f"Taux d'erreur grave : {(hospit_as_indemne/total_hospit)*100:.2f}%")

In [ ]:
# Masque : Prédit Tué (3) mais Réel était Hospitalisé (2) ou Léger (1)
y_pred = pipeline.predict(X_test)
faux_morts = X_test[(y_pred == 3) & (y_test < 3)]

In [ ]:
print("Caractéristiques des 'Faux Morts' :")
print(f"\nVitesse moyenne (VMA) : {faux_morts['vma'].mean():.1f} km/h")
print(f"Proportion de ceintures : {faux_morts['ceinture'].mean():.2f}")
print(f"Log-densité de population moyenne : {faux_morts['densite_log'].mean():.2f}")
print(f"Proportion hors agglomération :{(faux_morts['agg'] == 1).mean() * 100}")
# Compare avec la moyenne globale
print("\nCaractéristiques globales des accidents:")
print(f"\nVitesse moyenne globale : {X_test['vma'].mean():.1f} km/h")
print(f"Proportion de ceintures globale : {X_test['ceinture'].mean():.2f}")
print(f"Log-densité de population moyenne globale: {X_test['densite_log'].mean():.2f}")
print(f"Proportion hors agglomération globale :{(X_test['agg'] == 1).mean() * 100}")


Il semble que les variables les plus importantes dans la classification ont eu une influence dans le fait de caractériser les faux morts.

### Analyse avec véhicules

Observons maintenant les résultats obtenus en ajoutant le dataset véhicules.

In [ ]:
categorical_features_engv = [
    "place", "catu", "sexe", "trajet", "locp", "actp", "etatp", "age_bin", 
    "catr", "circ", "vosp", "prof", "plan", "larrout", "surf", "infra", 
    "situ", "mois", "jour", "lum", "dep", "agg", "int", "atm", "col",
    "catv", "senc", "obs", "obsm", "choc", "manv", "motor"
]

numerical_features_engv = [
    'vma', 'nbv', 'nb_param_lieux', 'densite_log', 'hour_sin', 'hour_cos', 
    'vma_densite_ratio', 'vma_nbv_ratio', 'ceinture', 'casque', 'airbag', 
    'safety', 'route_danger', 'int_danger', 'est_fragile', 'trajet_haute_grav', 
    'seul_avec_animal', 'surface_critique', 'meteo_critique', 'anomalie_atm', 
    'anomalie_surf', 'not_lum', 'rural_isole', 'ped_loc_seul',
    'mass_proxy', 
    'kinetic_energy_score', 
    'vulnerable_outside_agg', 
    'extreme_fixed_obs_impact', 
    'curve_drift_risk', 
    'side_impact_junction', 
    'night_fixed_obs', 
    'heavy_vs_vulnerable_collision'
]

In [ ]:
Xv = df_cleaned23v[numerical_features_engv + categorical_features_engv]
yv = df_cleaned23v['grav_ord']

X_trainv, X_testv, y_trainv, y_testv = train_test_split(Xv, yv, test_size=0.2, stratify=yv, random_state=42)

# n_jobs est à fixer en fonction du nombre de coeurs du CPU qu'on souhaite utiliser. Par défaut, ils sont tous utilisés.
pipelinev = get_pipeline(categorical_features_engv,n_jobs=15)

In [ ]:
pipelinev.fit(X_trainv,y_trainv)

In [ ]:
cmv = evaluate_predictions(pipeline,X_testv,y_testv,labels)

In [ ]:
from sklearn.metrics import classification_report
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cmv, display_labels=labels)
disp.plot(cmap='viridis', ax=ax, values_format='.2f')

plt.title("Matrice de Confusion Normalisée\n(Capacité de détection par classe de gravité)")
plt.show()

Nos résultats s'améliorent, avec une augmentation du f1-score pondéré de 4 points de pourcentage. 79% des usagers tués sont classés comme hospitalisés et tués contre 76% précédemment, et seuls 11% des blessés hospitalisés sont classés comme indemnes.

La précision de la classe Tué à bondi de 15% à 20%, ce qui est une forte progression dans la réduction de faux positifs. Les données du dataset véhicules sont particulièrement importantes pour prédire les chances de décès.

Cependant, le modèle a toujours des difficultés à distinguer les blessés légers des blessés hospitalisés. Cela peut être lié à des facteurs non présents dans notre base : fragilités spécifiques, antécédents médicaux, caractéristiques spécifiques des usagers absentes de l'analyse.

Quelles sont les features avec la plus grande importance dans notre modèle ?

In [ ]:
feature_importance_df = prediction.get_feature_importance_df(pipeline,categorical_features_eng,numerical_features_eng)

plt.figure(figsize=(12, 10))
sns.barplot(data=feature_importance_df.head(15), x='Importance', y='Feature', palette='magma')
plt.title("Top 15 des variables les plus déterminantes pour la gravité")
plt.xlabel("Importance (Gini Importance)")
plt.ylabel("Variables")
plt.tight_layout()
plt.show()

On peut voir l'apport majeur des variables de masse relative, de point de choc, et de manoeuvre. La violence intrinsèque du choc est un des premiers moteurs de gravité, devant les facteurs environnementaux seuls.

Bien que la classe Tué soit extrêmement minoritaire (2.61% des cas), le modèle parvient à  identifier près de 40% des cas, et les classifie à 80% dans blessés graves ou tués. La précision de la classe a augmenté de 1/3 grâce à l'ajout des véhicules, montrant encore une fois que les caractéristiques des véhicules impliqués sont des marqueurs de létalité particulièrement importants.

In [ ]:
print(round((df_cleaned23v["grav_ord"]==3).mean()*100,2))

## Prédiction 2005-2018

In [ ]:
categorical_features_eng05 = [
    "place", "catu", "sexe", "trajet", "locp", "actp", "etatp", "age_bin", 
    "catr", "circ", "vosp", "prof", "plan", "larrout", "surf", "infra", 
    "situ", "mois", "jour", "lum", "dep_clean", "agg", "int", "atm", "col",
    "catv", "senc", "obs", "obsm", "choc", "manv","env1"
]

numerical_features_eng05 = [
    'nbv', 'nb_param_lieux', 'densite_log', 'hour_sin', 'hour_cos', 
    'ceinture', 'casque', 'airbag', 
    'safety', 'route_danger', 'int_danger', 'est_fragile', 'trajet_haute_grav', 
    'seul_avec_animal', 'surface_critique', 'meteo_critique', 'anomalie_atm', 
    'anomalie_surf', 'not_lum', 'rural_isole', 'ped_loc_seul',
    'mass_proxy', 
    'vulnerable_outside_agg', 
    'extreme_fixed_obs_impact', 
    'curve_drift_risk', 
    'side_impact_junction', 
    'night_fixed_obs', 
    'heavy_vs_vulnerable_collision'
]

In [ ]:
X_old = df_cleaned_old[numerical_features_eng05 + categorical_features_eng05]
y_old = df_cleaned_old['grav_ord']

X_train_old, X_test_old, y_train_old, y_test_old = train_test_split(X_old, y_old, test_size=0.2, stratify=y_old, random_state=42)

 # n_jobs est à fixer en fonction du nombre de coeurs du CPU qu'on souhaite utiliser. Par défaut, ils sont tous utilisés.
pipeline_old = get_pipeline(categorical_features_eng05,n_jobs=15)

In [ ]:
# run_cross_val(pipeline_old,X_train_old,y_train_old,n_jobs=15)

In [ ]:
pipeline_old.fit(X_train_old, y_train_old)

# 12 min de chargement

In [ ]:
labels = [
    '0-Indemne', 
    '1-Léger', 
    '2-Hospitalisé', 
    '3-Tué'
]

cm_old = evaluate_predictions(pipeline_old,X_test_old,y_test_old,labels)

In [ ]:
from sklearn.metrics import classification_report
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_old, display_labels=labels)
disp.plot(cmap='viridis', ax=ax, values_format='.2f')

plt.title("Matrice de Confusion Normalisée\n(Capacité de détection par classe de gravité)")
plt.show()

In [ ]:
feature_importance_df_old = prediction.get_feature_importance_df(pipeline_old,categorical_features_eng05,numerical_features_eng05)

plt.figure(figsize=(12, 10))
sns.barplot(data=feature_importance_df_old.head(15), x='Importance', y='Feature', palette='magma')
plt.title("Top 15 des variables les plus déterminantes pour la gravité")
plt.xlabel("Importance (Gini Importance)")
plt.ylabel("Variables")
plt.tight_layout()
plt.show()

Ces résultats montrent un comportement du modèle différent de celui sur la période récente. L'absence de VMA a forcé le Random Forest à se rabattre sur d'autres signaux, ce qui modifie radicalement les socres de recall et l'ordre d'importance des features dans la prédiction. 

Le modèle identifie 63% des tués, bien plus que sur le dataset 2019-2024 ! En l'absence de VMA, le modèle s'appuie sur des variables très discriminantes comme la localisation (agg, rural_isole), la densité de population, ou la masse des véhicules.

Par contre, la précision chute à 14%. Il se trompe souvent avec 86% de faux positifs sur la mort. Il prédit bien trop souvent la mort.

Les hospitalisations sont beaucoup moins bien captées (36% contre 53%, et 38% de classifications en indemne ou blessé léger).

Cela suggère que la vitesse maximale autorisée et les features construites à partir d'elle étaient des prédicteurs cruciaux pour distinguer le blessé léger du blessé grave. Sans cette donnée, le modèle a tendance à plus "hésiter" et basculer vers le décès ou le léger.

Les scores généraux restent robustes, avec un f1-score pondéré de 0.61.

# Conclusion générale


Ce projet visait à modéliser la gravité des accidents de la route en France à travers l'exploitation des bases de données BAAC. En comparant deux périodes distinctes (2005-2018 et 2019-2024), nous avons pu mesurer l'impact de l'évolution des données collectées sur la performance des modèles de prédiction.

L'utilisation du Random Forest a permis d'atteindre une précision globale de 65% et un f1-score global de 65% sur les données récentes. L'intégration de la table véhicules a permis d'augmenter ces métriques de 4 points par rapport à un modèles basé uniquement sur les circonstances et sur les caractéristiques générales des usagers.

- Sur la période historique (sans VMA), le modèle est plus alarmiste avec un recall des décès à 63%, mais une plus faible précision.

- Sur la période récente (avec VMA), le modèle devient plus nuancé et précis, prouvant que la vitesse autorisée est un pivot important pour distinguer le degré du coût humain des accidents.

La création de certaines cross-features (ceinture, safety, densite, casque, rural_isole, kinetic_energy_score, vma_densite_ratio...) a permis de contourner le caractère boîte noire du Random Forest en permettant une meilleure interprétation des résultats.

Malgré ces avancées, la zone grise entre blessés légers et blessés hospitalisés reste difficile à modéliser. Cela souligne une limite intrinsèque de la base BAAC : l'absence de données sur le profil et l'état de santé intrinsèque des victimes (état physiologique, antécédents, propension à conduire dangereusement) et sur la violence réelle de l'impact (vitesse instantanée au choc).

Si ce projet a permis de dresser un état des lieux prédictif, plusieurs pistes s'ouvrent pour prolonger la réflexion.

Le potentiel des coordonnées GPS est important. Avec des données plus nombreuses et de meilleure qualité dans les années à venir, nous pourrions croiser les accidents avec des points d'intérêt spécifiques, ce qui permettrait de créer un outil de cartographie des points géographiques les plus dangereux pour les accidents.

D'ici quelques années, les bases BAAC pourraient s'enrichir des données issues des "boîtes noires" des véhicules. On pourrait alors étudier comment les aides à la conduite modifient la hiérachie des variables de gravité. 

En conclusion, notre travail souligne que l'intelligence du modèle réside davantage dans sa capacité à intégrer des réalités physiques et contextuelles. Cela confirme que la modernisation des données est un levier indispensable pour mieux anticiper le risque et, à terme, mieux protéger les usagers.